# Model Development
## Week 4 — Training Three Models
This notebook trains three models for the Healthcare Claims Intelligence Suite:
- **Model 1:** Claim Denial Predictor (Logistic Regression + Random Forest)
- **Model 2:** Anomaly Detector (Isolation Forest)
- **Model 3:** ICD Code Classifier (Naive Bayes + TF-IDF)

In [12]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

os.chdir('/Users/darshanbalaji/Documents/healthcare-claims-ai/healthcare-claims-ai')

# Load processed data
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Denial rate in training set:", y_train.mean().round(4))

X_train shape: (40000, 42)
X_test shape: (10000, 42)
Denial rate in training set: 0.2646


## Model 1a: Claim Denial Predictor — Logistic Regression
Logistic Regression chosen as the primary model for interpretability.
Coefficients directly show which features drive denial probability —
a regulatory requirement in healthcare AI.

In [13]:
# Train Logistic Regression
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=500,
    random_state=42
)

lr.fit(X_train, y_train)

# Quick check
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr))

# Save model
os.makedirs('models', exist_ok=True)
joblib.dump(lr, 'models/denial_predictor_lr.pkl')
print("Logistic Regression saved")

              precision    recall  f1-score   support

           0       0.82      0.66      0.73      7354
           1       0.39      0.59      0.47      2646

    accuracy                           0.64     10000
   macro avg       0.60      0.63      0.60     10000
weighted avg       0.70      0.64      0.66     10000

Logistic Regression saved


## Model 1b: Claim Denial Predictor — Random Forest
Random Forest trained for comparison against Logistic Regression.
Expected to have higher accuracy but lower interpretability.

In [14]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))

joblib.dump(rf, 'models/denial_predictor_rf.pkl')
print("Random Forest saved")

              precision    recall  f1-score   support

           0       0.75      0.82      0.78      7354
           1       0.34      0.26      0.29      2646

    accuracy                           0.67     10000
   macro avg       0.55      0.54      0.54     10000
weighted avg       0.64      0.67      0.65     10000

Random Forest saved


## Model Tuning — Denial Predictor
Attempting to improve recall on the denied class from 0.60 toward 0.80 target.
Two approaches: (1) GridSearchCV for hyperparameter tuning, 
(2) Decision threshold adjustment.

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, make_scorer

# GridSearchCV to find best parameters
recall_scorer = make_scorer(recall_score, pos_label=1)

params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [200, 500, 1000]
}

grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', random_state=42),
    params,
    scoring=recall_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validated recall:", grid.best_score_.round(4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'C': 10, 'max_iter': 200, 'solver': 'liblinear'}
Best cross-validated recall: 0.5941


In [19]:
from sklearn.metrics import recall_score, precision_score

# Train best model from grid search
lr_tuned = LogisticRegression(
    C=10, 
    max_iter=200, 
    solver='liblinear',
    class_weight='balanced',
    random_state=42
)
lr_tuned.fit(X_train, y_train)
y_prob_tuned = lr_tuned.predict_proba(X_test)[:, 1]

# Test different thresholds
print(f"{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'F1':<8}")
print("-" * 45)

for threshold in [0.5, 0.4, 0.3, 0.25, 0.2]:
    y_pred_thresh = (y_prob_tuned >= threshold).astype(int)
    recall = recall_score(y_test, y_pred_thresh)
    precision = precision_score(y_test, y_pred_thresh)
    f1 = 2 * (precision * recall) / (precision + recall)
    print(f"{threshold:<12} {recall:<10.3f} {precision:<12.3f} {f1:<8.3f}")

Threshold    Recall     Precision    F1      
---------------------------------------------
0.5          0.591      0.385        0.467   
0.4          0.683      0.370        0.480   
0.3          1.000      0.265        0.418   
0.25         1.000      0.265        0.418   
0.2          1.000      0.265        0.418   


## Model 2: Anomaly Detector — Isolation Forest
Unsupervised anomaly detection. No labels used in training.
Flags claims that are statistically unusual compared to the rest of the dataset.

In [ ]:
anomaly_features = [
    'billed_to_median_ratio',
    'high_cost_outlier', 
    'provider_denial_rate',
    'duplicate_flag',
    'filing_delay_bucket_61+'
]

X_anomaly = X_train[anomaly_features]
scaler = StandardScaler()
X_anomaly_scaled = scaler.fit_transform(X_anomaly)

iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_anomaly_scaled)

X_full_scaled = scaler.transform(X_anomaly)
scores = iso.decision_function(X_full_scaled)
labels = iso.predict(X_full_scaled)

print(f"Anomalies detected: {(labels == -1).sum()}")
print(f"Normal claims: {(labels == 1).sum()}")

joblib.dump(iso, 'models/anomaly_detector.pkl')
joblib.dump(scaler, 'models/anomaly_scaler.pkl')
print("Isolation Forest saved")

## Tuning Results and Decision

GridSearchCV identified C=10, solver=liblinear as optimal parameters 
but only marginally improved recall from 0.60 to 0.594.

Threshold adjustment at 0.4 gives the best practical balance:
- Recall: 0.683 — catches 68% of real denials
- Precision: 0.370 — acceptable false alarm rate for a pre-submission tool
- Threshold 0.3 and below predicts all claims as denied — not useful

**Final model:** Logistic Regression with C=10, threshold=0.4
**Recall gap note:** Target was 0.80 — falling short is attributed to 
synthetic data limitations. Real payer remittance data would provide 
stronger signal and likely close this gap.

In [15]:
# Select features for anomaly detection
anomaly_features = [
    'billed_to_median_ratio',
    'high_cost_outlier', 
    'provider_denial_rate',
    'duplicate_flag',
    'filing_delay_bucket_61+'
]

X_anomaly = X_train[anomaly_features]

# Scale features
scaler = StandardScaler()
X_anomaly_scaled = scaler.fit_transform(X_anomaly)

# Train Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_anomaly_scaled)

# Get anomaly scores on full dataset
X_full_anomaly = X_train[anomaly_features]
X_full_scaled = scaler.transform(X_full_anomaly)
scores = iso.decision_function(X_full_scaled)
labels = iso.predict(X_full_scaled)

print(f"Anomalies detected: {(labels == -1).sum()}")
print(f"Normal claims: {(labels == 1).sum()}")

joblib.dump(iso, 'models/anomaly_detector.pkl')
joblib.dump(scaler, 'models/anomaly_scaler.pkl')
print("Isolation Forest saved")

Anomalies detected: 2000
Normal claims: 38000
Isolation Forest saved


## Model 3: ICD Code Classifier — Naive Bayes + TF-IDF
Text classification model that maps plain-English diagnosis descriptions
to ICD-10 code categories. Uses TF-IDF to convert text to numerical 
features and Multinomial Naive Bayes for classification.

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

cms_df = pd.read_csv('data/raw/PHY_R26_P05_V10_D24_Prov_Svc.csv', 
                     nrows=500000,
                     low_memory=False)

# Use HCPCS descriptions as training text
# Create CPT category groups as target labels
def get_cpt_category(code):
    code = str(code)
    if code.startswith('99'):
        return 'Evaluation_Management'
    elif code[0] in ['7', '8']:
        return 'Radiology_Pathology'
    elif code[0] in ['2', '3']:
        return 'Surgery_Cardiology'
    elif code[0] in ['9'] and not code.startswith('99'):
        return 'Medicine_Procedures'
    else:
        return 'Other'

cms_df['cpt_category'] = cms_df['HCPCS_Cd'].apply(get_cpt_category)
print("CPT category distribution:")
print(cms_df['cpt_category'].value_counts())

CPT category distribution:
cpt_category
Evaluation_Management    158849
Radiology_Pathology      116134
Medicine_Procedures      103455
Other                    101226
Surgery_Cardiology        20336
Name: count, dtype: int64


In [22]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# Clean text
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = ''.join([c for c in text if c.isalpha() or c == ' '])
    text = ' '.join([w for w in text.split() if w not in stop_words])
    return text

cms_df['clean_desc'] = cms_df['HCPCS_Desc'].apply(clean_text)

# Train-test split for ICD classifier
from sklearn.model_selection import train_test_split

X_text = cms_df['clean_desc']
y_text = cms_df['cpt_category']

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text, y_text,
    test_size=0.2,
    stratify=y_text,
    random_state=42
)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf_train = vectorizer.fit_transform(X_text_train)
X_tfidf_test = vectorizer.transform(X_text_test)

print("TF-IDF matrix shape:", X_tfidf_train.shape)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/darshanbalaji/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


TF-IDF matrix shape: (400000, 5000)


In [23]:
# Train Naive Bayes
nb = MultinomialNB()
nb.fit(X_tfidf_train, y_text_train)

# Evaluate
y_text_pred = nb.predict(X_tfidf_test)
print(classification_report(y_text_test, y_text_pred))

# Top-3 accuracy
y_text_prob = nb.predict_log_proba(X_tfidf_test)
top3_correct = 0
for i, true_label in enumerate(y_text_test):
    top3_indices = y_text_prob[i].argsort()[-3:]
    top3_labels = nb.classes_[top3_indices]
    if true_label in top3_labels:
        top3_correct += 1

top3_accuracy = top3_correct / len(y_text_test)
print(f"\nTop-3 Accuracy: {top3_accuracy:.4f}")

# Save model and vectorizer
joblib.dump(nb, 'models/icd_classifier.pkl')
joblib.dump(vectorizer, 'models/tfidf_vectorizer.pkl')
print("ICD classifier saved")

                       precision    recall  f1-score   support

Evaluation_Management       1.00      1.00      1.00     31770
  Medicine_Procedures       1.00      1.00      1.00     20691
                Other       1.00      0.99      0.99     20245
  Radiology_Pathology       1.00      1.00      1.00     23227
   Surgery_Cardiology       0.98      0.99      0.98      4067

             accuracy                           1.00    100000
            macro avg       0.99      0.99      0.99    100000
         weighted avg       1.00      1.00      1.00    100000


Top-3 Accuracy: 0.9999
ICD classifier saved


## Honest Model Assessment

The denial predictor achieves 0.68 recall against a 0.80 target.
This gap is not closing with hyperparameter tuning or threshold 
adjustment, which indicates the limitation is in the data rather 
than the model architecture.

Synthetic denial labels generated from simple probability rules 
create a low-signal dataset where even a well-tuned model cannot 
reliably distinguish denied from approved claims beyond ~70% recall.

A production system trained on real payer remittance data with CARC 
codes as ground truth labels would provide substantially richer signal. 
This is the single highest-priority improvement for a production version 
of this system.

In [21]:
joblib.dump(lr_tuned, 'models/denial_predictor_lr.pkl')
print("Tuned model saved")

Tuned model saved
